### IMPORTAMOS LIBRERIAS Y CARGAMOS EL DATASET ORIGINAL

In [ ]:
import pandas as pd
import numpy as np

# Carga del dataset original
json = "../data/raw/streaming_users_dirty.json"
df_raw = pd.read_json(json)

print("=== Dataset original cargado ===")

=== Dataset original cargado ===


### TRABAJAMOS SOBRE UNA COPIA DEL DATASET ORIGINAL Y CONFIGURAMOS LA AUDITORIA

In [149]:
# Preservamos Dataset Original Y Configuramos Auditoría
df_copy  = df_raw.copy()
log = []

def registrar(df_copy, paso, desc):
    log.append({
        'Paso'         : paso,
        'Descripción'  : desc,
        'Filas'        : len(df_copy),
        'Nulos'        : df_copy.isnull().sum().sum(),
        'Retención (%)': round(len(df_copy) / 8160 * 100, 2)
    })

registrar(df_copy, 0, "Dataset original")

### PASO 1 - ELIMINACION DE FILAS DUPLICADAS

In [150]:
# --- Paso 1: Eliminación de filas duplicadas ---
filas_originales = len(df_copy)

# Aplicamos la eliminación omitiendo la columna 'age'
df_copy = df_copy.drop_duplicates(
    subset=df_copy.columns.difference(['age'])
).reset_index(drop=True)

filas_finales = len(df_copy)
filas_borradas = filas_originales - filas_finales

print("="*45)
print("PASO 1 - LIMPIEZA DE REGISTROS DUPLICADOS")
print("="*45)
print(f"• Filas iniciales:               {filas_originales}")
print(f"• Filas duplicadas detectadas:   {filas_borradas}")
print(f"• Filas limpias resultantes:     {filas_finales}")
print("="*45)

# Guardamos el progreso en el log
registrar(df_copy, 1, "Eliminación de duplicados")

PASO 1 - LIMPIEZA DE REGISTROS DUPLICADOS
• Filas iniciales:               8160
• Filas duplicadas detectadas:   128
• Filas limpias resultantes:     8032


Se eliminaron las 128 filas duplicadas identificadas mediante el análisis de consistencia (excluyendo la variable age). Al remover estas réplicas, el dataset se redujo de 8160 a 8032 registros limpios. Esta acción es fundamental para garantizar que cada fila represente un usuario único con su comportamiento real, evitando sesgos de dobles conteos de usuarios.

### PASO 2 - NORMALIZACION DE COLUMNAS CATEGORICAS

In [151]:
# --- Paso 2: Normalizacion de Columnas Categoricas ---
filas_iniciales = len(df_copy)

# Guardamos una copia temporal de las columnas antes de modificarlas
columnas_a_limpiar = ['subscription_plan', 'country', 'favorite_genre']

# Pasamos a minúsculas y quitamos espacios
for col in columnas_a_limpiar:
    df_copy[col] = df_copy[col].str.strip().str.lower()

mapa_subscription_plan = {
    'basico':'básico',
    'basic':'básico',
    'estandar':'estándar',
    'standard':'estándar',
    'std':'estándar',
    'premiun':'premium'                                          
}

mapa_country = {
    'arg':'argentina',
    'bra':'brasil',
    'brazil':'brasil',
    'chl':'chile',
    'col':'colombia',
    'mex':'méxico',
    'mexico':'méxico',
    'per':'perú',
    'peru':'perú',
    'ury':'uruguay',
}

mapa_favorite_genre = {
    'action':'acción',
    'accion':'acción',
    'comedy':'comedia',
    'crime':'crimen',
    'doc':'documental',
    'documentary':'documental',
    'thriler':'thriller'
}

df_copy['subscription_plan'] = df_copy['subscription_plan'].replace(mapa_subscription_plan)
df_copy['country'] = df_copy['country'].replace(mapa_country)
df_copy['favorite_genre'] = df_copy['favorite_genre'].replace(mapa_favorite_genre)

filas_finales = len(df_copy)

print("\n" + "="*47)
print("PASO 2 - NORMALIZACIÓN DE COLUMNAS CATEGÓRICAS")
print("="*47)
print(f"• Filas iniciales:            {filas_iniciales}")
print(f"• Filas resultantes:          {filas_finales}")
print("="*47)

# Guardamos en el log
registrar(df_copy, 2, "Normalización Categoricas")


PASO 2 - NORMALIZACIÓN DE COLUMNAS CATEGÓRICAS
• Filas iniciales:            8032
• Filas resultantes:          8032


Se realizó la normalización y estandarización de las variables categóricas subscription_plan, country y favorite_genre. El proceso consistió en unificar los textos a minúsculas, remover espacios en blanco y aplicar diccionarios de mapeo personalizados para corregir errores de tipeo, variantes idiomáticas (inglés/español) y abreviaturas. Como resultado, se consolidaron categorías limpias y consistentes sin alterar el volumen de datos (manteniendo las 8032 filas),.

### PASO 3: NORMALIZACION DE COLUMNA last_login_date

In [152]:
# --- Paso 3: Normalizacion de columna last_login_date ---
filas_iniciales = len(df_copy)

# 1. Reemplazamos las fechas raras usando np.nan
fechas_raras = ['not_available', '0000-00-00', 'null', '']
df_copy['last_login_date'] = df_copy['last_login_date'].replace(fechas_raras, np.nan)

# 2. Convertimos a datetime
df_copy['last_login_date'] = pd.to_datetime(df_copy['last_login_date'], format='mixed', dayfirst=True, errors='coerce')

# 3. Guardamos para saber cuántas se van por nulos
filas_antes_dropna = len(df_copy)
df_copy = df_copy.dropna(subset=["last_login_date"])
filas_eliminadas_nulos = filas_antes_dropna - len(df_copy)

# 4. Filtramos para retener solo las fechas entre 2018 - 2025
filas_antes_rango = len(df_copy)
mask_rango_valido = (df_copy["last_login_date"].dt.year >= 2018) & (df_copy["last_login_date"].dt.year <= 2025)
df_copy = df_copy[mask_rango_valido]
filas_eliminadas_rango = filas_antes_rango - len(df_copy)

# 5. Convertimos la columna a datetime
df_copy['last_login_date'] = df_copy['last_login_date'].astype('datetime64[ns]')

filas_finales = len(df_copy)
total_borradas = filas_iniciales - filas_finales

print("="*45)
print("PASO 3 - FILTRADO Y LIMPIEZA DE FECHAS")
print("="*45)
print(f"• Filas iniciales en este paso:  {filas_iniciales}")
print(f"• Total de filas eliminadas:     {total_borradas}")
print(f"  └─ Borradas por nulas/rotas:   {filas_eliminadas_nulos}")
print(f"  └─ Borradas fuera de rango:    {filas_eliminadas_rango}")
print(f"• Filas limpias resultantes:     {filas_finales}")
print("="*45)

# Guardamos el progreso
registrar(df_copy, 3, "Limpieza de fechas y filtrado de periodo 2018-2025")

PASO 3 - FILTRADO Y LIMPIEZA DE FECHAS
• Filas iniciales en este paso:  8032
• Total de filas eliminadas:     399
  └─ Borradas por nulas/rotas:   364
  └─ Borradas fuera de rango:    35
• Filas limpias resultantes:     7633


Se realizó el procesamiento y la depuración de fechas de la columna last_login_date. El proceso se dividió en dos etapas: en primer lugar, se transformó la variable al tipo de dato adecuado (datetime64), lo que permitió identificar y remover 364 registros con valores nulos o textos corruptos (como 'not_available' o formatos vacíos). En segundo lugar, se aplicó una máscara lógica para descartar 35 registros que presentaban fechas inconsistentes fuera del período establecido (2018-2025). Como resultado de esta limpieza, el dataset se redujo a 7633.

### DIAGNÓSTICO DE FALTA EN MINUTOS Y GENERO

In [153]:
# ── DIAGNÓSTICO DE FALTA EN MINUTOS Y GENERO ──

# Creamos las columnas temporales de control
df_copy['minutos_falta'] = df_copy['monthly_watch_time_mins'].isnull().astype(int)
df_copy['genero_falta'] = df_copy['favorite_genre'].isnull().astype(int)

print("========================================================")
print("         ¿Por qué faltan los minutos mensuales?")
print("========================================================")
tasa_falta_por_plan = df_copy.groupby('subscription_plan')['minutos_falta'].mean() * 100
print("\n🔹 Porcentaje de faltantes en minutos según el PLAN:")
print(tasa_falta_por_plan.map("{:.2f}%".format).to_string())

tasa_falta_por_pais = df_copy.groupby('country')['minutos_falta'].mean() * 100
print("\n🔹 Porcentaje de faltantes en minutos según el PAÍS:")
print(tasa_falta_por_pais.map("{:.2f}%".format).to_string())


print("\n=========================================================")
print("          ¿Por qué falta el género favorito?")
print("=========================================================")
tasa_genero_por_plan = df_copy.groupby('subscription_plan')['genero_falta'].mean() * 100
print("\n🔹 Porcentaje de faltantes en género según el PLAN:")
print(tasa_genero_por_plan.map("{:.2f}%".format).to_string())

tasa_genero_por_pais = df_copy.groupby('country')['genero_falta'].mean() * 100
print("\n🔹 Porcentaje de faltantes en género según el PAÍS:")
print(tasa_genero_por_pais.map("{:.2f}%".format).to_string())

print("=========================================================")

# Borramos ambas columnas temporales
df_copy = df_copy.drop(columns=['minutos_falta', 'genero_falta'])

         ¿Por qué faltan los minutos mensuales?

🔹 Porcentaje de faltantes en minutos según el PLAN:
subscription_plan
básico      0.12%
estándar    1.22%
premium     9.68%

🔹 Porcentaje de faltantes en minutos según el PAÍS:
country
argentina    2.07%
brasil       2.53%
chile        2.16%
colombia     1.92%
méxico       2.83%
perú         2.50%
uruguay      2.85%

          ¿Por qué falta el género favorito?

🔹 Porcentaje de faltantes en género según el PLAN:
subscription_plan
básico      3.25%
estándar    3.04%
premium     2.70%

🔹 Porcentaje de faltantes en género según el PAÍS:
country
argentina    2.82%
brasil       3.25%
chile        3.33%
colombia     2.84%
méxico       3.38%
perú         3.06%
uruguay      2.76%


* monthly_watch_time_mins (Mecanismo MAR): Existe un sesgo sistemático asociado al plan de suscripción. Los usuarios con plan premium presentan una tasa de faltantes muy elevada (9.68%) en comparación con el plan básico (0.12%). La ausencia del dato está condicionada por la variable subscription_plan.

* favorite_genre (Mecanismo MCAR): La ausencia de datos es uniforme y se distribuye de manera homogénea (cercana al 3%) a lo largo de todos los países y planes de suscripción. Al no encontrarse correlación con otras variables, se determina que la falta es completamente al azar.

### PASO 4 - IMPUTACION DE FALTANTES Y NULOS

In [154]:
# --- Paso 4: Imputacion de faltantes y nulos ---
filas_iniciales = len(df_copy)

# Contamos cuántos nulos hay en cada columna ANTES de rellenar
nulos_genero = df_copy['favorite_genre'].isnull().sum()
nulos_minutos = df_copy['monthly_watch_time_mins'].isnull().sum()

# 1. Para el género creamos la categoría 'No especificado'
df_copy['favorite_genre'] = df_copy['favorite_genre'].fillna('No especificado')

# 2. Rellenamos los nulos de minutos con la mediana por plan
df_copy['monthly_watch_time_mins'] = df_copy['monthly_watch_time_mins'].fillna(
    df_copy.groupby('subscription_plan')['monthly_watch_time_mins'].transform('median')
)

filas_finales = len(df_copy)

print("="*45)
print("PASO 4 - IMPUTACIÓN DE VALORES FALTANTES")
print("="*45)
print(f"• Filas iniciales en este paso:   {filas_iniciales}")
print(f"  └─ Géneros completados:         {nulos_genero}")
print(f"  └─ Minutos mensuales (mediana): {nulos_minutos}")
print(f"• Filas resultantes:              {filas_finales}")
print("="*45)

# Guardamos el progreso
registrar(df_copy, 4, "Imputación de faltantes en género y tiempo mensual")

PASO 4 - IMPUTACIÓN DE VALORES FALTANTES
• Filas iniciales en este paso:   7633
  └─ Géneros completados:         234
  └─ Minutos mensuales (mediana): 184
• Filas resultantes:              7633


* favorite_genre: Dado que los 234 nulos respondían a un mecanismo puramente al azar (MCAR), se optó por una imputarlos como 'No especificado'. Esto evita inventar preferencias y preserva la neutralidad de los perfiles.

* monthly_watch_time_mins: Como los 184 faltantes presentaban un mecanismo sesgado según el plan (MAR), se utilizó una imputación por agrupación (mediana por plan). Esto garantiza que los usuarios Premium reciban valores realistas y acordes a su segmento, en lugar de un promedio global que distorsionaría la distribución.

### Deteccion de Outliers en Columnas Numericas

In [155]:
# Deteccion de Outliers en Columnas Numericas
columnas_numericas = ['age', 'monthly_watch_time_mins', 'customer_support_tickets']

print("=== DETECCIÓN DE OUTLIERS ===")

for col in columnas_numericas:
    # 1. Calculamos cuartiles y el rango intercuartílico
    Q1 = df_copy[col].quantile(0.25)
    Q3 = df_copy[col].quantile(0.75)
    IQR = Q3 - Q1
    
    # 2. Definimos límites (Criterios: Edad > 1 y < 100, IQR para el resto)
    if col == 'age':
        lim_inf = 1.0   # Aplicamos restricción: mayor a 1 año
        lim_sup = 100.0 # Aplicamos restricción: menor a 100 años
    else:
        lim_inf_matematico = Q1 - 3 * IQR
        lim_sup = Q3 + 3 * IQR
        lim_inf = max(0, lim_inf_matematico)
    
    # 3. Contamos cuántos están fuera del rango establecido
    total_outliers = ((df_copy[col] < lim_inf) | (df_copy[col] > lim_sup)).sum()

    print(f"\n📌 Columna: {col}")
    print(f"  -> Rango lógico razonable : entre {lim_inf:.1f} y {lim_sup:.1f}")
    print(f"  -> Outliers detectados    : {total_outliers}")
    
    # Muestra los extremos si se detectaron anomalías
    if total_outliers > 0:
        print(f"  -> Valores más BAJOS en el dataset: {df_copy[col].nsmallest(5).values}")
        print(f"  -> Valores más ALTOS en el dataset: {df_copy[col].nlargest(5).values}")

=== DETECCIÓN DE OUTLIERS ===

📌 Columna: age
  -> Rango lógico razonable : entre 1.0 y 100.0
  -> Outliers detectados    : 94
  -> Valores más BAJOS en el dataset: [-5 -5 -5 -5 -5]
  -> Valores más ALTOS en el dataset: [150 150 150 150 150]

📌 Columna: monthly_watch_time_mins
  -> Rango lógico razonable : entre 0.0 y 2762.4
  -> Outliers detectados    : 178
  -> Valores más BAJOS en el dataset: [-120. -120. -120. -120. -120.]
  -> Valores más ALTOS en el dataset: [99999. 99999. 99999. 99999. 99999.]

📌 Columna: customer_support_tickets
  -> Rango lógico razonable : entre 0.0 y 4.0
  -> Outliers detectados    : 104
  -> Valores más BAJOS en el dataset: [-1 -1 -1 -1 -1]
  -> Valores más ALTOS en el dataset: [150 150 150 150 150]


Se combinaron criterios para la deteccion de outliers, para la edad se utilizo el rango que va de 1 a 100 años, para los minutos y tickets de soporte se utilizo el Rango Intercuartílico con k = 3. Con esto se pudo identificar los límites aceptables y cuantificar las anomalías en las variables analizadas:
* age: El cálculo estadístico fija un límite superior de 94 años. Se detectaron 94 outliers, confirmando de esta forma valores erróneos de -5 y 150 años en el dataset.
* monthly_watch_time_mins: El límite razonable se estableció en 2762.4 minutos. Entre los 178 outliers detectados se encuentran registros con el valores como -120 y 99999 minutos, lo cual es imposible tener minutos negativos o mas de 43200 minutos (suponiendo que una persona se paso todo el mes viendo streaming 60x24x30).
* customer_support_tickets: El umbral estadístico arrojó un máximo de 4 tickets. Se detectaron 104 registros donde se observan usuarios con tickets negativos o una cantidad exesiva (-1 o 150 tickets).

La presencia de valores idénticos y repetitivos tanto en los mínimos como en los máximos confirma que no estamos ante usuarios con comportamientos extremos, sino ante fallos de carga o datos corruptos del sistema.

### PASO 5 - TRATAMIENTO DE OUTLIERS

In [156]:
# --- Paso 5: Tratamiento de Outliers ---
filas_iniciales = len(df_copy)

# --- 1. EDAD: Imputación por Mediana según el Plan de Suscripción ---
medianas_edad_por_plan = df_copy.groupby('subscription_plan')['age'].median()

# Contamos cuántas filas estan fuera de rango (< 1 o > 100)
edades_fuera_rango = ((df_copy['age'] < 1.0) | (df_copy['age'] > 100.0)).sum()

# Aplicamos la imputación por mediana segun plan
df_copy['age'] = df_copy.apply(
    lambda row: medianas_edad_por_plan[row['subscription_plan']] 
    if row['age'] < 1.0 or row['age'] > 100.0 
    else row['age'], 
    axis=1
)

df_copy['age'] = df_copy['age'].astype('Int64')

# --- 2. MINUTOS: Clipping Estadístico Extremo (3.0xIQR) ---
Q1_mins = df_copy['monthly_watch_time_mins'].quantile(0.25)
Q3_mins = df_copy['monthly_watch_time_mins'].quantile(0.75)
IQR_mins = Q3_mins - Q1_mins
lim_sup_mins = Q3_mins + 3.0 * IQR_mins

# Contamos cuántos registros superan el límite superior (incluye los negativos en el lower y los 99999 en el upper)
minutos_suavizados = ((df_copy['monthly_watch_time_mins'] < 0) | (df_copy['monthly_watch_time_mins'] > lim_sup_mins)).sum()
df_copy['monthly_watch_time_mins'] = df_copy['monthly_watch_time_mins'].clip(lower=0, upper=lim_sup_mins)

# --- 3. TICKETS: Clipping Estadístico Extremo (3.0xIQR) ---
Q1_tix = df_copy['customer_support_tickets'].quantile(0.25)
Q3_tix = df_copy['customer_support_tickets'].quantile(0.75)
IQR_tix = Q3_tix - Q1_tix
lim_sup_tix = Q3_tix + 3.0 * IQR_tix

# Contamos cuántos registros tenían ráfagas artificiales o tickets negativos
tickets_suavizados = ((df_copy['customer_support_tickets'] < 0) | (df_copy['customer_support_tickets'] > lim_sup_tix)).sum()
df_copy['customer_support_tickets'] = df_copy['customer_support_tickets'].clip(lower=0, upper=lim_sup_tix)

filas_finales = len(df_copy)

# 📊 Reporte unificado y elegante para tu consola corporativa
print("="*45)
print("PASO 5 - TRATAMIENTO OUTLIERS")
print("="*45)
print(f"• Filas iniciales:                  {filas_iniciales}")
print(f"  └─ Edades corregidas por plan:    {edades_fuera_rango}")
print(f"  └─ Minutos corregidos:            {minutos_suavizados}")
print(f"  └─ Tickets corregidos:            {tickets_suavizados}")
print(f"• Filas resultantes:                {filas_finales}")
print("="*45)

# Guardamos el progreso
registrar(df_copy, 5, "Tratamiento de outliers")

PASO 5 - TRATAMIENTO OUTLIERS
• Filas iniciales:                  7633
  └─ Edades corregidas por plan:    94
  └─ Minutos corregidos:            178
  └─ Tickets corregidos:            104
• Filas resultantes:                7633


Se sanearon las variables numéricas aplicando estrategias de imputación y suavizado (clipping), eliminando el ruido extremo sin perder registros:
* age (Imputación por Plan): Se corrigieron los 94 perfiles con edades imposibles (como -5 y 150 años) reemplazándolos por la mediana de edad de su respectivo plan de suscripción.
* monthly_watch_time_mins (3.0 x IQR): Se suavizaron los 178 registros atípicos, absorbiendo los valores negativos (-120) y positivos extremos (99999) al tope matemático de 2762.4 minutos, protegiendo así a los usuarios de alto consumo real.
* customer_support_tickets (3.0 x IQR): Se normalizaron los 104 casos (valores de -1 y 150), limitándolos al máximo estadístico de 4.0 tickets.

### Paso 6 - Exportacion de log pipeline y dataset limpio. Resumen final

In [157]:
# ── Resumen Final ──

# 1. Transformamos la lista de diccionarios en un DataFrame limpio para el log
log_df = pd.DataFrame(log)
log_df['Retención (%)'] = (log_df['Filas'] / len(df_raw) * 100).round(2)
log_df['Retención (%)'] = log_df['Retención (%)'].map("{:.2f}%".format)

print("\n" + "="*55)
print("HISTORIAL COMPLETO DE AUDITORÍA (LOG ETL)")
print("="*55)
print(log_df.to_string(index=False))
print("="*55)

# Exportación del log etl
log_df.to_csv("../logs/pipeline_log.csv", index=False, encoding='utf-8')

# Exportación del dataset procesado y limpio
df_copy.to_json('../data/processed/streaming_users_clean.json', orient='records', indent=2, date_format='iso')

# 2. Resumen Métricas Finales
retencion_final = (len(df_copy) / len(df_raw)) * 100

print("\n" + "="*29)
print(" RESUMEN DE MéTRICAS FINALES")
print("="*29)
print(f"• Usuarios iniciales : {len(df_raw)}")
print(f"• Usuarios finales   : {len(df_copy)}")
print(f"• Filas eliminadas   : {len(df_raw) - len(df_copy)}")
print(f"• Retención de datos : {retencion_final:.2f}%")
print("="*47)
print("Archivos guardados con éxito:")
print("  └─ Log ETL:  ../logs/pipeline_log.csv")
print("  └─ Dataset:  ../data/processed/streaming_users_clean.json")
print("="*47)


HISTORIAL COMPLETO DE AUDITORÍA (LOG ETL)
 Paso                                        Descripción  Filas  Nulos Retención (%)
    0                                   Dataset original   8160    753       100.00%
    1                          Eliminación de duplicados   8032    753        98.43%
    2                          Normalización Categoricas   8032    753        98.43%
    3 Limpieza de fechas y filtrado de periodo 2018-2025   7633    418        93.54%
    4 Imputación de faltantes en género y tiempo mensual   7633      0        93.54%
    5                            Tratamiento de outliers   7633      0        93.54%

 RESUMEN DE MéTRICAS FINALES
• Usuarios iniciales : 8160
• Usuarios finales   : 7633
• Filas eliminadas   : 527
• Retención de datos : 93.54%
Archivos guardados con éxito:
  └─ Log ETL:  ../logs/pipeline_log.csv
  └─ Dataset:  ../data/processed/streaming_users_clean.json


#### Conclusión del Proceso de Calidad y Limpieza:
* Alta Retención (93.54%): De 8160 usuarios iniciales se preservaron 7633. La baja del 6.46% (527 filas) se concentró únicamente en la eliminacion de duplicados (Paso 1) y fechas rotas o fuera de rango (Paso 3).
* Cero Nulos: Se eliminaron los 753 faltantes iniciales sin borrar filas (Paso 4), usando imputaciones inteligentes como 'No especificado' en género y la mediana por plan en minutos mensuales.
* Control de Outliers (Paso 5): Se sanearon los 376 registros atípicos (edades imposibles, errores 99999 y tickets extremos) mediante clipping e imputación por plan, estabilizando las distribuciones numéricas sin perder un solo dato.